# 03 — Model Training

Three-phase EfficientNet-B3 training pipeline:

- **Phase 1**: Head-only with light augmentation (frozen backbone)
- **Phase 2**: Partial unfreeze (last 2 feature blocks) with full augmentation + Mixup
- **Phase 3**: Full unfreeze with full augmentation + Mixup

Key features:
- Patient-level stratified splitting (no data leakage)
- WeightedRandomSampler for class balancing (no class-weighted loss to avoid double-rebalancing)
- Ordinal-aware cross-entropy loss (aligns with QWK metric)
- Mixed-precision training (AMP)
- Checkpointing on Quadratic Weighted Kappa
- ONNX export for production deployment

In [ ]:
import importlib
import os
import sys

IS_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/diabetic_retinopathy"
    %pip install -q timm albumentations kaggle
else:
    PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import src.config
importlib.reload(src.config)
from src import config
from src.data_manager import ensure_dataset

ensure_dataset()
config.ensure_dirs()

print(f"Images dir: {config.IMAGES_DIR} (exists: {os.path.isdir(config.IMAGES_DIR)})")
print(f"Outputs dir: {config.OUTPUTS_DIR}")

In [ ]:
import pandas as pd

from src import config
from src.augmentations import get_light_train_transforms, get_train_transforms, get_valid_transforms
from src.dataset import (
    compute_class_weights,
    create_dataloaders,
    load_and_split_dataframe,
    preprocess_images_to_cache,
    validate_cache_for_dataframe,
)
from src.model import build_model, count_parameters
from src.trainer import Trainer
from src.utils import get_device, seed_everything

seed_everything()
device = get_device()

## 1. Data Preparation

In [ ]:
train_df, valid_df, test_df = load_and_split_dataframe()

print(f'Train : {len(train_df):,} images  ({train_df["patient_id"].nunique():,} patients)')
print(f'Valid : {len(valid_df):,} images  ({valid_df["patient_id"].nunique():,} patients)')
print(f'Test  : {len(test_df):,} images  ({test_df["patient_id"].nunique():,} patients)')
print(f'\nTrain class distribution:')
print(train_df["level"].value_counts().sort_index())

## 2. Preprocessing Cache + DataLoaders

Images are preprocessed once (crop + CLAHE at 384x384) and cached to disk.
We create **two sets of loaders**:
- `loaders_phase1`: light augmentation for head-only training
- `loaders_phase23`: full augmentation for fine-tuning phases

In [ ]:
# Build preprocessing cache (runs once, skips if already cached with matching config)
all_splits_df = pd.concat(
    [
        train_df[["image", "level"]],
        valid_df[["image", "level"]],
        test_df[["image", "level"]],
    ],
    ignore_index=True,
).drop_duplicates(subset=["image"])

cached_images_dir = preprocess_images_to_cache(
    df=all_splits_df,
    source_images_dir=config.IMAGES_DIR,
    img_size=config.IMG_SIZE,
    cache_image_format=config.CACHE_IMAGE_FORMAT,
    cache_image_quality=config.CACHE_IMAGE_QUALITY,
    fail_on_missing=True,
)

cache_summary = validate_cache_for_dataframe(
    df=all_splits_df,
    cache_dir=cached_images_dir,
    cache_image_format=config.CACHE_IMAGE_FORMAT,
)

print(f"\nCache dir: {cached_images_dir}")
print(f"Cache validation: expected={cache_summary['expected']} present={cache_summary['present']}")

In [ ]:
# Phase 1 loaders: light augmentation
light_transforms = get_light_train_transforms(config.IMG_SIZE)
valid_transforms = get_valid_transforms(config.IMG_SIZE)

loaders_phase1 = create_dataloaders(
    train_df=train_df,
    valid_df=valid_df,
    test_df=test_df,
    train_transform=light_transforms,
    valid_transform=valid_transforms,
    images_dir=cached_images_dir,
    batch_size=config.PHASE1_BATCH_SIZE,
    preprocessing=False,
    use_sampler=config.USE_WEIGHTED_SAMPLER,
    fail_on_missing=True,
)

# Phase 2/3 loaders: full augmentation
full_transforms = get_train_transforms(config.IMG_SIZE)

loaders_phase23 = create_dataloaders(
    train_df=train_df,
    valid_df=valid_df,
    test_df=test_df,
    train_transform=full_transforms,
    valid_transform=valid_transforms,
    images_dir=cached_images_dir,
    batch_size=config.PHASE2_BATCH_SIZE,
    preprocessing=False,
    use_sampler=config.USE_WEIGHTED_SAMPLER,
    fail_on_missing=True,
)

print(f"Phase 1 train batches: {len(loaders_phase1['train'])}")
print(f"Phase 2/3 train batches: {len(loaders_phase23['train'])}")
print(f"Valid batches: {len(loaders_phase1['valid'])}")
print(f"Test batches: {len(loaders_phase1['test'])}")

## 3. Model + Class Weights

In [ ]:
model = build_model(pretrained=True)
params = count_parameters(model)

print(f"Model: {config.MODEL_NAME}")
print(f"Input size: {config.IMG_SIZE}x{config.IMG_SIZE}")
print(f"Total parameters: {params['total']:,}")
print(f"Dropout rate: {config.DROPOUT_RATE}")
print(f"\nLoss: {'OrdinalCrossEntropy' if config.USE_ORDINAL_LOSS else 'LabelSmoothingCE'}")
print(f"Label smoothing: {config.LABEL_SMOOTHING}")
print(f"Mixup (Phase 2/3): {config.USE_MIXUP} (alpha={config.MIXUP_ALPHA})")
print(f"WeightedRandomSampler: {config.USE_WEIGHTED_SAMPLER}")
print(f"Class-weighted loss: {config.USE_CLASS_WEIGHTED_LOSS}")

In [ ]:
# Class weights: only used if config.USE_CLASS_WEIGHTED_LOSS is True
# The Trainer constructor checks this flag and sets weights to None otherwise
class_weights = compute_class_weights(train_df["level"].values)
print("Class weights (for reference):")
for name, w in zip(config.CLASS_NAMES, class_weights):
    print(f"  {name:20s}: {w:.4f}")

if not config.USE_CLASS_WEIGHTED_LOSS:
    print("\n(Class weights NOT applied to loss — using WeightedRandomSampler only)")

## 4. Training

| Phase | Backbone | Augmentation | LR | Epochs | Mixup |
|-------|----------|-------------|-----|--------|-------|
| 1 | Frozen | Light | 3e-4 | 8 | No |
| 2 | Last 2 blocks | Full | 5e-5 | 15 | Yes |
| 3 | All | Full | 2e-5 | 25 | Yes |

In [ ]:
trainer = Trainer(
    model=model,
    device=device,
    class_weights=class_weights,
)

history = trainer.fit(
    train_loader=loaders_phase23["train"],
    valid_loader=loaders_phase23["valid"],
    train_loader_light=loaders_phase1["train"],
)

## 5. Training Curves

In [ ]:
from src.evaluate import plot_training_curves

plot_training_curves(
    history,
    save_path=os.path.join(config.FIGURES_DIR, "training_curves.png"),
)

## 6. Quick Validation Check

In [ ]:
from src.evaluate import compute_metrics, predict

trainer.load_best_checkpoint("phase3")

y_true, y_pred, y_proba = predict(model, loaders_phase23["valid"], device)
metrics = compute_metrics(y_true, y_pred, y_proba)

print(f"Validation QWK      : {metrics['qwk']:.4f}")
print(f"Validation Accuracy : {metrics['accuracy']:.4f}")
print(f"Validation ROC-AUC  : {metrics['roc_auc_weighted']:.4f}")
print(f"\n{metrics['classification_report_str']}")

## 7. Save Test Split for Evaluation Notebook

In [ ]:
test_csv_path = os.path.join(config.OUTPUTS_DIR, "test_split.csv")
test_df.to_csv(test_csv_path, index=False)
print(f"Test split saved to {test_csv_path} ({len(test_df)} images)")

## 8. ONNX Export

In [ ]:
from src.export import export_to_onnx

onnx_path = export_to_onnx(
    checkpoint_path=os.path.join(config.CHECKPOINTS_DIR, "best_model_phase3.pth"),
)
print(f"\nONNX model exported to: {onnx_path}")